In [19]:
import gzip
import ast
import json
import urllib.request
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
DATA_DIR = Path("../data/raw/amazon_intersection")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("../data/processed/amazon_intersection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def download_if_missing(url, path):
    path = Path(path)

    if not path.exists():
        print(f"Scarico: {url}")
        urllib.request.urlretrieve(url, path)
        print(f"Salvato in: {path}")
    else:
        print(f"File già presente: {path}")

In [20]:
QA_BEAUTY_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/icdm/QA_Beauty.json.gz"
QA_BEAUTY_PATH = DATA_DIR / "QA_Beauty.json.gz"

download_if_missing(QA_BEAUTY_URL, QA_BEAUTY_PATH)
with gzip.open(QA_BEAUTY_PATH, "rt", encoding="utf-8") as f:
    first_row = ast.literal_eval(next(f))

print(first_row.keys())
first_row

def load_qa_asins(path):
    asins = set()
    n_rows = 0

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in tqdm(f, desc="Leggo QA Beauty"):
            line = line.strip()

            if not line:
                continue

            row = ast.literal_eval(line)

            asin = row.get("asin")

            if asin:
                asins.add(asin)

            n_rows += 1

    return asins, n_rows


qa_asins, qa_rows = load_qa_asins(QA_BEAUTY_PATH)

print(f"Righe QA Beauty: {qa_rows:,}")
print(f"Prodotti unici nel QA Beauty: {len(qa_asins):,}")

File già presente: ..\data\raw\amazon_intersection\QA_Beauty.json.gz
dict_keys(['asin', 'questions'])


Leggo QA Beauty: 5857it [00:02, 2482.63it/s]

Righe QA Beauty: 5,857
Prodotti unici nel QA Beauty: 5,857


In [21]:
meta_all_beauty = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True
)

meta_all_beauty_asins = set(meta_all_beauty["parent_asin"])

print(f"Prodotti metadata 2023 All_Beauty: {len(meta_all_beauty_asins):,}")

intersection_all_beauty = qa_asins & meta_all_beauty_asins

print(f"Prodotti QA Beauty: {len(qa_asins):,}")
print(f"Prodotti metadata 2023 All_Beauty: {len(meta_all_beauty_asins):,}")
print(f"Intersezione QA Beauty ∩ All_Beauty metadata 2023: {len(intersection_all_beauty):,}")

print(f"% QA coperto da All_Beauty 2023: {len(intersection_all_beauty) / len(qa_asins) * 100:.2f}%")
print(f"% All_Beauty 2023 coperto da QA: {len(intersection_all_beauty) / len(meta_all_beauty_asins) * 100:.4f}%")


Prodotti metadata 2023 All_Beauty: 112,590
Prodotti QA Beauty: 5,857
Prodotti metadata 2023 All_Beauty: 112,590
Intersezione QA Beauty ∩ All_Beauty metadata 2023: 46
% QA coperto da All_Beauty 2023: 0.79%
% All_Beauty 2023 coperto da QA: 0.0409%


In [22]:
meta_beauty_pc = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Beauty_and_Personal_Care",
    split="full",
    trust_remote_code=True
)

meta_beauty_pc_asins = set(meta_beauty_pc["parent_asin"])

print(f"Prodotti metadata 2023 Beauty_and_Personal_Care: {len(meta_beauty_pc_asins):,}")
intersection_beauty_pc = qa_asins & meta_beauty_pc_asins

print(f"Prodotti QA Beauty: {len(qa_asins):,}")
print(f"Prodotti metadata 2023 Beauty_and_Personal_Care: {len(meta_beauty_pc_asins):,}")
print(f"Intersezione QA Beauty ∩ Beauty_and_Personal_Care metadata 2023: {len(intersection_beauty_pc):,}")

print(f"% QA coperto da Beauty_and_Personal_Care 2023: {len(intersection_beauty_pc) / len(qa_asins) * 100:.2f}%")
print(f"% Beauty_and_Personal_Care 2023 coperto da QA: {len(intersection_beauty_pc) / len(meta_beauty_pc_asins) * 100:.4f}%")
results = pd.DataFrame([
    {
        "reviews_2023_category": "All_Beauty",
        "qa_category": "Beauty",
        "qa_products": len(qa_asins),
        "metadata_products": len(meta_all_beauty_asins),
        "intersection_products": len(intersection_all_beauty),
        "qa_coverage_%": len(intersection_all_beauty) / len(qa_asins) * 100,
        "metadata_coverage_%": len(intersection_all_beauty) / len(meta_all_beauty_asins) * 100,
    },
    {
        "reviews_2023_category": "Beauty_and_Personal_Care",
        "qa_category": "Beauty",
        "qa_products": len(qa_asins),
        "metadata_products": len(meta_beauty_pc_asins),
        "intersection_products": len(intersection_beauty_pc),
        "qa_coverage_%": len(intersection_beauty_pc) / len(qa_asins) * 100,
        "metadata_coverage_%": len(intersection_beauty_pc) / len(meta_beauty_pc_asins) * 100,
    }
])

results

Prodotti metadata 2023 Beauty_and_Personal_Care: 1,028,914
Prodotti QA Beauty: 5,857
Prodotti metadata 2023 Beauty_and_Personal_Care: 1,028,914
Intersezione QA Beauty ∩ Beauty_and_Personal_Care metadata 2023: 2,620
% QA coperto da Beauty_and_Personal_Care 2023: 44.73%
% Beauty_and_Personal_Care 2023 coperto da QA: 0.2546%


,reviews_2023_category,qa_category,qa_products,metadata_products,intersection_products,qa_coverage_%,metadata_coverage_%
0,All_Beauty,Beauty,5857,112590,46,0.785385,0.040856
1,Beauty_and_Personal_Care,Beauty,5857,1028914,2620,44.732798,0.254637


In [23]:
if len(intersection_beauty_pc) > len(intersection_all_beauty):
    best_category = "Beauty_and_Personal_Care"
    best_intersection = intersection_beauty_pc
    best_metadata = meta_beauty_pc
else:
    best_category = "All_Beauty"
    best_intersection = intersection_all_beauty
    best_metadata = meta_all_beauty

print(f"Categoria 2023 migliore: {best_category}")
print(f"Prodotti comuni QA + metadata 2023: {len(best_intersection):,}")

Categoria 2023 migliore: Beauty_and_Personal_Care
Prodotti comuni QA + metadata 2023: 2,620


In [24]:
reviews_dataset_name = f"raw_review_{best_category}"

reviews_stream = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    reviews_dataset_name,
    split="full",
    trust_remote_code=True,
    streaming=True
)

OUT_REVIEWS_PATH = OUTPUT_DIR / f"reviews_2023_{best_category}_QA_Beauty_common.jsonl.gz"
OUT_COUNTS_PATH = OUTPUT_DIR / f"review_counts_2023_{best_category}_QA_Beauty_common.csv"

review_count_by_asin = Counter()
users_common = set()

n_seen = 0
n_kept = 0

with gzip.open(OUT_REVIEWS_PATH, "wt", encoding="utf-8") as out:
    for row in tqdm(reviews_stream, desc=f"Filtro reviews 2023 {best_category}"):
        n_seen += 1

        asin = row.get("parent_asin")

        if asin in best_intersection:
            filtered_row = {
                "user_id": row.get("user_id"),
                "parent_asin": row.get("parent_asin"),
                "rating": row.get("rating"),
                "title": row.get("title"),
                "text": row.get("text"),
                "timestamp": row.get("timestamp"),
                "helpful_vote": row.get("helpful_vote"),
                "verified_purchase": row.get("verified_purchase"),
            }

            out.write(json.dumps(filtered_row, ensure_ascii=False) + "\n")

            review_count_by_asin[asin] += 1

            if row.get("user_id"):
                users_common.add(row.get("user_id"))

            n_kept += 1

print(f"Review lette totali: {n_seen:,}")
print(f"Review salvate sui prodotti comuni: {n_kept:,}")
print(f"Prodotti comuni con almeno una review: {len(review_count_by_asin):,}")
print(f"Utenti unici nei prodotti comuni: {len(users_common):,}")
print(f"File review filtrate salvato in: {OUT_REVIEWS_PATH}")

Filtro reviews 2023 Beauty_and_Personal_Care: 23911390it [16:38, 23955.03it/s]

Review lette totali: 23,911,390
Review salvate sui prodotti comuni: 615,015
Prodotti comuni con almeno una review: 2,619
Utenti unici nei prodotti comuni: 543,992
File review filtrate salvato in: ..\data\processed\amazon_intersection\reviews_2023_Beauty_and_Personal_Care_QA_Beauty_common.jsonl.gz


In [25]:
review_counts = (
    pd.Series(review_count_by_asin)
    .rename_axis("parent_asin")
    .reset_index(name="n_reviews")
    .sort_values("n_reviews", ascending=False)
)

review_counts.to_csv(OUT_COUNTS_PATH, index=False)

print(f"File conteggi salvato in: {OUT_COUNTS_PATH}")

review_counts.describe()
print("Prodotti comuni QA + metadata:", len(best_intersection))
print("Prodotti comuni con almeno 1 review:", (review_counts["n_reviews"] >= 1).sum())
print("Prodotti comuni con almeno 5 review:", (review_counts["n_reviews"] >= 5).sum())
print("Prodotti comuni con almeno 10 review:", (review_counts["n_reviews"] >= 10).sum())
print("Prodotti comuni con almeno 20 review:", (review_counts["n_reviews"] >= 20).sum())

print("Review totali salvate:", review_counts["n_reviews"].sum())
print("Utenti unici:", len(users_common))
products_with_reviews = set(review_count_by_asin.keys())

final_valid_asins = best_intersection & products_with_reviews

print(f"Prodotti con QA + metadata + review: {len(final_valid_asins):,}")
VALID_ASINS_PATH = OUTPUT_DIR / f"valid_asins_2023_{best_category}_QA_Beauty_final.txt"

with open(VALID_ASINS_PATH, "w", encoding="utf-8") as f:
    for asin in sorted(final_valid_asins):
        f.write(asin + "\n")

print(f"ASIN finali salvati in: {VALID_ASINS_PATH}")
metadata_rows = []

for row in tqdm(best_metadata, desc="Filtro metadata finali"):
    asin = row.get("parent_asin")

    if asin in final_valid_asins:
        metadata_rows.append(row)

metadata_final = pd.DataFrame(metadata_rows)

OUT_METADATA_PATH = OUTPUT_DIR / f"metadata_2023_{best_category}_QA_Beauty_final.csv"
metadata_final.to_csv(OUT_METADATA_PATH, index=False)

print(f"Metadata finali salvati in: {OUT_METADATA_PATH}")
print(metadata_final.shape)

metadata_final.head()

File conteggi salvato in: ..\data\processed\amazon_intersection\review_counts_2023_Beauty_and_Personal_Care_QA_Beauty_common.csv
Prodotti comuni QA + metadata: 2620
Prodotti comuni con almeno 1 review: 2619
Prodotti comuni con almeno 5 review: 2382
Prodotti comuni con almeno 10 review: 2191
Prodotti comuni con almeno 20 review: 1951
Review totali salvate: 615015
Utenti unici: 543992
Prodotti con QA + metadata + review: 2,619
ASIN finali salvati in: ..\data\processed\amazon_intersection\valid_asins_2023_Beauty_and_Personal_Care_QA_Beauty_final.txt


Filtro metadata finali:   0%|          | 0/1028914 [00:00<?, ?it/s]

Filtro metadata finali: 100%|██████████| 1028914/1028914 [01:32<00:00, 11165.76it/s]


Metadata finali salvati in: ..\data\processed\amazon_intersection\metadata_2023_Beauty_and_Personal_Care_QA_Beauty_final.csv
(2619, 16)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Health & Personal Care,100 Grams Bag of Natural Indigo Powder (wasma ...,3.8,44,"[Indigo Powder ( wasma in Arabic and Urdu ), 1...",[Indigo powder is combined with Henna to creat...,None,"{'hi_res': [None, None, None], 'large': ['http...","{'title': [], 'url': [], 'user_id': []}",All India Store,"[Beauty & Personal Care, Hair Care, Hair Color...","{""Color"": ""Orange,Blue"", ""Package Information""...",B00312PRU2,None,None,None
1,All Beauty,Relogy Natural Acne Treatment Kit (60 Day) for...,3.2,140,"[Everything you need for clear, balanced skin....","[Often times, our best efforts at clear our po...",None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",RELOGY,"[Beauty & Personal Care, Skin Care, Face, Sets...","{""Is Discontinued By Manufacturer"": ""No"", ""Pro...",B00CWJZJAS,None,None,None
2,All Beauty,"Premier Dead Sea Classic BIOX serum, Total Age...",4.2,187,[✅TESTED HYPOALLERGENIC + Dermatologically tes...,[],114.99,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['Premier Dead Sea Classic Miracle N...,Premier by Dead Sea Premier,"[Beauty & Personal Care, Skin Care, Face, Trea...","{""Skin Type"": ""Sensitive Skin,All Skin Types"",...",B004HFNFU6,None,None,None
3,All Beauty,SHANY Boutique 40 Colors Neon Highly Pigmented...,4.1,2195,[Set of 40 bold and vivid eyeshadow colors. Ki...,[The SHANY Boutique Eye shadow Palette (Set of...,12.13,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['Style Code Live: Buzzfeed Beauty T...,SHANY,"[Beauty & Personal Care, Makeup, Makeup Palettes]","{""Color"": ""Boutique 40 Color"", ""Brand"": ""SHANY...",B001C6H6F8,None,None,None
4,All Beauty,Avon Reversalist Day Renewal Cream SPF 25 1.7o...,4.1,377,"[SPF 25 UVA/UVB, Keeping skin looking younger,...",[Avon ANEW Reversalist Day Cream SPF 25 will i...,None,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['Clientele Elastology Lotus Story']...,AVON,"[Beauty & Personal Care, Skin Care, Face, Crea...","{""Brand"": ""AVON"", ""Item Form"": ""Cream"", ""Unit ...",B0034UBHNM,None,None,None


In [26]:
reviews_final = pd.read_json(
    OUT_REVIEWS_PATH,
    lines=True,
    compression="gzip"
)

reviews_final.head()

,user_id,parent_asin,rating,title,text,timestamp,helpful_vote,verified_purchase
0,AFEIN7QWSQ6EFW45LKD7ZOCSMLGQ,B0000536P4,1,"This is not the real face cream, after using ...","This is not the real face cream, after using i...",2018-06-29 15:57:13.127,1,True
1,AH6CATODIVPVUOJEWHRSRCSKAOHA,B00GLE8SNK,1,One Star,Been using this for a while and it did not do ...,2015-03-17 16:53:14.000,1,True
2,AH6PLOGWYIVIWLJTY756BHNFD4YA,B00018KV1E,1,This is the second bottle of Angel I have purc...,This is the second bottle of Angel I have purc...,2014-10-17 11:13:53.000,19,True
3,AEYFU5FQAJYXFREC4KHQ34EGZ3LQ,B0050INH2U,1,One Star,"NOT MY favorite lotion, not very good, I'm an ...",2017-11-05 21:45:47.061,0,True
4,AEYFU5FQAJYXFREC4KHQ34EGZ3LQ,B00147FGJ8,5,Five Stars,EXCELLENT LOTION!! LOVE IT,2017-09-03 22:10:55.354,0,True


In [27]:
ratings_final = reviews_final[[
    "user_id",
    "parent_asin",
    "rating",
    "timestamp"
]].copy()

ratings_final = ratings_final.rename(columns={
    "parent_asin": "item_id"
})

OUT_RATINGS_PATH = OUTPUT_DIR / f"ratings_2023_{best_category}_QA_Beauty_final.csv"
ratings_final.to_csv(OUT_RATINGS_PATH, index=False)

print(f"Ratings finali salvati in: {OUT_RATINGS_PATH}")
print(ratings_final.shape)

ratings_final.head()

Ratings finali salvati in: ..\data\processed\amazon_intersection\ratings_2023_Beauty_and_Personal_Care_QA_Beauty_final.csv
(615015, 4)


,user_id,item_id,rating,timestamp
0,AFEIN7QWSQ6EFW45LKD7ZOCSMLGQ,B0000536P4,1,2018-06-29 15:57:13.127
1,AH6CATODIVPVUOJEWHRSRCSKAOHA,B00GLE8SNK,1,2015-03-17 16:53:14.000
2,AH6PLOGWYIVIWLJTY756BHNFD4YA,B00018KV1E,1,2014-10-17 11:13:53.000
3,AEYFU5FQAJYXFREC4KHQ34EGZ3LQ,B0050INH2U,1,2017-11-05 21:45:47.061
4,AEYFU5FQAJYXFREC4KHQ34EGZ3LQ,B00147FGJ8,5,2017-09-03 22:10:55.354


In [28]:
summary = {
    "qa_category": "Beauty",
    "reviews_2023_category": best_category,
    "qa_products": len(qa_asins),
    "metadata_products": len(meta_beauty_pc_asins) if best_category == "Beauty_and_Personal_Care" else len(meta_all_beauty_asins),
    "qa_metadata_intersection_products": len(best_intersection),
    "final_products_with_qa_metadata_reviews": len(final_valid_asins),
    "final_reviews": len(ratings_final),
    "final_users": ratings_final["user_id"].nunique(),
    "avg_reviews_per_product": len(ratings_final) / len(final_valid_asins) if len(final_valid_asins) > 0 else 0,
}

summary_df = pd.DataFrame([summary])

OUT_SUMMARY_PATH = OUTPUT_DIR / "intersection_summary.csv"
summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

summary_df

,qa_category,reviews_2023_category,qa_products,metadata_products,qa_metadata_intersection_products,final_products_with_qa_metadata_reviews,final_reviews,final_users,avg_reviews_per_product
0,Beauty,Beauty_and_Personal_Care,5857,1028914,2620,2619,615015,543992,234.828179
